## **Setup**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import time
import math
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import copy

# Local
from chatGnT.config import CFG, ensure_dirs
from chatGnT.data import load, preprocess, tokenize
from chatGnT.models import transformer, positional_encoding, train, evaluate, predict
import chatGnT.utils as utils

ensure_dirs(CFG)

In [ ]:
# Load models


In [ ]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel2Head(
    ntoken_amt=len(vocab_amt),
    ntoken_ingred=len(vocab_ingred),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
# Note warning:
# UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is
# False because encoder_layer.self_attn.batch_first was not True (use
# # batch_first for better inference performance)

pad_id_amt = vocab_amt["<pad>"]
pad_id_ingred = vocab_ingred["<pad>"]
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion_amt = torch.nn.CrossEntropyLoss(ignore_index=pad_id_amt)
criterion_ingred = torch.nn.CrossEntropyLoss(ignore_index=pad_id_ingred)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)

## **Run with Test User Input**

#### **Single Head Model**

In [28]:
input_mod = [("<amt>1 oz</amt>", "<ingred>gin</ingred>")]
predict.predict(best_model, device, pad_id, vocab, inv_vocab, input_mod)

#### **Double Head Model**

In [49]:
input_mod = [("<amt>1 oz</amt>", "<ingred>gin</ingred>")]
predict.predict_2head(best_model, device, pad_id_amt, pad_id_ingred, vocab_amt, vocab_ingred, inv_vocab_amt, inv_vocab_ingred, input_mod)


[('<amt>1 oz</amt>', '<ingred>gin</ingred>'),
 ('<amt>1 oz</amt>', '<ingred>orange-bitters</ingred>'),
 ('<amt>add</amt>', '<ingred>vanilla-extract</ingred>'),
 ('<end>', '<ingred>water</ingred>')]

## **Visualize Embeddings**